# 프롬프트 실험실 (Prompt Lab)

운영 파이프라인과 완전히 분리된 노트북입니다. GitHub에 아무것도 push하지 않습니다.

사용 방법:
1. 런타임 → 런타임 유형 변경 → T4 GPU 선택
2. 셀 1, 2를 한 번만 실행 (모델 로드, 1~2분)
3. 셀 3의 `PROMPT`, `NEGATIVE_PROMPT`, `STEPS`, `GUIDANCE` 값을 자유롭게 바꿔가며
   반복 실행 → 결과를 즉시 확인 (모델은 이미 로드되어 있어 매번 1초 내)
4. 마음에 드는 조합을 찾으면 그 `PROMPT` 문구를 그대로 복사해서 전달해주세요.
   (Claude가 prompt_templates.json에 반영합니다)

In [ ]:
# 1. 환경 설정
!pip install diffusers transformers accelerate -q

import torch
from diffusers import AutoPipelineForText2Image

print("환경 설정 완료")

In [ ]:
# 2. 모델 로드 (1~2분, 한 번만 실행)
pipe = AutoPipelineForText2Image.from_pretrained(
    "stabilityai/sdxl-turbo", torch_dtype=torch.float16, variant="fp16"
).to("cuda")
print("모델 로드 완료")

In [ ]:
# 3. 실험 셀 - 아래 값들을 자유롭게 바꿔가며 반복 실행하세요
#    셀 1,2는 한 번만 실행하면 됩니다. 이 셀만 반복 실행하면 됩니다.
import requests, time

# GitHub에서 config 읽기 (NEGATIVE_PROMPT 기본값용)
try:
    cfg = requests.get("https://raw.githubusercontent.com/stockpipeline/stockpipeline/main/config.json").json()
    DEFAULT_NEGATIVE = cfg.get("image", {}).get("negative_prompt", "")
except:
    DEFAULT_NEGATIVE = "multiple bowls, many small dishes, grid of plates, banchan spread, tiled composition, repeated pattern"

# ── 여기를 수정하세요 ────────────────────────────────────
PROMPT = (
    "A single steaming bowl of Korean yukgaejang spicy beef soup, "
    "one bowl only, no other dishes, top-down view, "
    "professional food photography, white background"
)

NEGATIVE_PROMPT = DEFAULT_NEGATIVE  # config에서 자동으로 가져옴 (또는 직접 수정)

STEPS = 4          # 1~4 (SDXL-Turbo 권장, 높을수록 디테일↑ 속도↓)
GUIDANCE = 1.5     # 0=negative_prompt 무효, 1~3 권장
NUM_IMAGES = 4     # 한 번에 생성할 장수
# ────────────────────────────────────────────────────────

print(f"PROMPT: {PROMPT[:80]}...")
print(f"NEGATIVE: {NEGATIVE_PROMPT[:60]}...")
print(f"steps={STEPS}, guidance={GUIDANCE}, num={NUM_IMAGES}")
print()

start = time.time()
for i in range(NUM_IMAGES):
    seed = torch.randint(0, 2**31 - 1, (1,)).item()
    generator = torch.Generator(device="cuda").manual_seed(seed)
    image = pipe(
        prompt=PROMPT,
        negative_prompt=NEGATIVE_PROMPT,
        num_inference_steps=STEPS,
        guidance_scale=GUIDANCE,
        width=1024,
        height=1024,
        generator=generator,
    ).images[0]
    print(f"  [{i+1}/{NUM_IMAGES}] seed={seed}, {time.time()-start:.1f}s")
    display(image)

print(f"\n완료! 총 {time.time()-start:.1f}초")
print("마음에 드는 이미지가 있으면 그 PROMPT 문구를 복사해서 Claude에게 전달해주세요.")